# Engine: Noether Currents — sigma = 1/2 is derived, never assigned

**File:** `ValaQuenta/noether.py`
**Wiki:** [wiki/noether.md](../../wiki/noether.md)

The claim this notebook tests: from **any** starting position and **any**
energy, the mathematics forces sigma = 1/2. If that is true, sigma = 1/2 is a
result. If it had to be set, it would be an assumption.

The derivation:

```
From the right (sigma > 1/2):  F(sigma) = exp(-sigma*E)
From the left  (sigma < 1/2):  B(sigma) = exp(-(1-sigma)*E)
They meet where F = B:  -sigma = -(1-sigma)  =>  sigma = 1/2
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
import math, cmath
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('python', sys.version.split()[0])

In [ ]:
from ValaQuenta import NoetherCurrents, SemanticWord

N = NoetherCurrents()

print(f'{"E":>10} {"sigma_0":>10} {"forced_sigma":>24} {"|s-0.5|":>12}')
for E in [0.5, 1.0, 2.0, 10.0]:
    for s0 in [0.0, 0.25, 0.9, -3.0]:
        s = N.forced_sigma(E, s0)
        print(f'{E:10.1f} {s0:10.2f} {s!r:>24} {abs(s-0.5):12.3e}')

For `E <= 10` every row lands on 0.5, from four starting positions
including a negative one. That is the claim, and in this range it holds.

## Where it stops holding

`wiki/noether.md` and `README.md` both record
`forced_sigma(E=100.0, sigma_0=0.0) -> 0.500000000000`. **That does not
reproduce.** The cell below runs the documented case.

In [ ]:
s = N.forced_sigma(100.0, 0.0)
print(f'forced_sigma(E=100.0, sigma_0=0.0) = {s!r}')
print(f'wiki/README record                 = 0.500000000000')
print(f'reproduces                         = {abs(s - 0.5) < 1e-9}')

### Why

`forced_sigma` is a fixed-point iteration, not a root solve:

```
F = exp(-sigma*E)
B = exp(-(1-sigma)*E)
sigma_new = (F*sigma + B*(1-sigma)) / (F + B)
```

This is a softmax-weighted average of `sigma` and `1-sigma`.

- **Small E**: `F` and `B` are both close to 1, the weights are nearly equal,
  and the average collapses to `(sigma + (1-sigma))/2 = 1/2` immediately.
- **Large E**: the exponentials differ by many orders of magnitude. For
  `sigma < 1/2`, `F >> B`, so `sigma_new -> sigma`. The step size falls below
  the `1e-12` tolerance on the first iteration and the loop breaks, **returning
  `sigma_0` essentially unchanged**. The guard `if F + B < 1e-30: break` does
  the same thing when both exponentials underflow.

So the loop does not fail loudly — it exits early and returns its own input.

The analytic derivation in the docstring is not in question: `F = B` implies
`exp(-sigma*E) = exp(-(1-sigma)*E)` implies `sigma = 1 - sigma` implies
`sigma = 1/2`, for every `E > 0`. **The mathematics is sound; the iteration
used to demonstrate it is not a reliable solver.** The trailing comment
`return sigma   # always 0.5` is incorrect as written.

In [ ]:
# Map the boundary. Nothing is tuned here -- this is a plain sweep.
Es = np.logspace(-2, 2.2, 60)
sigmas, converged = [], []
for E in Es:
    v = N.forced_sigma(float(E), 0.0)
    sigmas.append(v)
    converged.append(abs(v - 0.5) < 1e-6)

first_bad = next((E for E, c in zip(Es, converged) if not c), None)
print(f'converges to 1/2 for E up to ~{first_bad:.2f} (sigma_0 = 0.0)')
print(f'{sum(converged)}/{len(Es)} sampled energies converge')
print()
for E in [10.0, 15.0, 20.0, 25.0, 30.0, 50.0, 100.0]:
    v = N.forced_sigma(E, 0.0)
    print(f'  E={E:<7} sigma={v!r:<24} half={abs(v-0.5)<1e-6}')

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.semilogx(Es, sigmas, 'o-', ms=3, color='#3f7fb0', label='forced_sigma')
ax.axhline(0.5, color='#b04a3f', lw=1, ls='--', label='sigma = 1/2 (claimed)')
if first_bad:
    ax.axvline(first_bad, color='#2f8f5f', ls=':', lw=1.2,
               label=f'convergence lost ~E={first_bad:.1f}')
ax.set_xlabel('E'); ax.set_ylabel('forced_sigma(E, 0.0)')
ax.set_title('the iteration converges to 1/2 only for small E')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### The overflow

For `sigma_0 < 0` and large `E`, `exp(-sigma*E)` is `exp(+large)` and the
function raises rather than returning anything. Kept here rather than removed,
because a caller passing a negative starting sigma is exactly the "from ANY
starting position" case the docstring advertises.

In [ ]:
for E, s0 in [(100.0, -3.0), (1000.0, -3.0), (1e4, -3.0), (1e4, 0.9)]:
    try:
        print(f'  E={E:<9} sigma_0={s0:<6} -> {N.forced_sigma(E, s0)!r}')
    except Exception as exc:
        print(f'  E={E:<9} sigma_0={s0:<6} -> {exc.__class__.__name__}: {exc}')

### What this does and does not affect

`sigma = 1/2` is derived independently elsewhere in the repo — by the geometry
in `hamiltonian.py` (RedBlue balance), and empirically by `understand.py`,
which derives it per word without going through this iteration. Those are not
disturbed by this.

What is affected is the specific numerical demonstration in this file, and the
two documentation lines that quote its `E=100` output.

## The three-phase balance

The boundary is **oriented**: up toward the next Cayley-Dickson shadow, down
toward the zero divisors. Not forward and backward in time.

```
J_up   = E = x*p          (away from ZD, toward the next CD level)
J_down = -J_up            (toward ZD, toward collapse)
J_3    = (J_up - J_down)/2 = E
```

In [ ]:
w = SemanticWord('tree')
print('word:', w)
f = N.forward(w); b = N.backward(w); r = N.rotating_field(w)
print(f'  forward  J_up   = {f!r}')
print(f'  backward J_down = {b!r}')
print(f'  rotating J_3    = {r!r}')
print(f'  balance         = {N.balance(w)!r}')
print()
print('sigma = 1/2 is the shadow of the world above falling on the world below.')
print('C projects onto R at sigma=1/2. H projects onto C at sigma=1/2.')